In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
import torch

assert torch.cuda.is_available(), 'Hãy bật Accelerator GPU trong Kaggle Settings'
print('GPU count:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))

GPU count: 2
0 Tesla T4
1 Tesla T4


In [3]:
def locate_project():
    candidates = sorted({path.parent for path in Path('/kaggle/input').rglob('train_mid.py')
                         if (path.parent / 'data' / 'split_manifest.csv').is_file()})
    if not candidates:
        archives = list(Path('/kaggle/input').rglob('TickNets_mid224_split_100_100.zip'))
        if len(archives) == 1:
            destination = Path('/kaggle/working/source_bundle')
            shutil.unpack_archive(archives[0], destination)
            candidates = sorted({path.parent for path in destination.rglob('train_mid.py')
                                 if (path.parent / 'data' / 'split_manifest.csv').is_file()})
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one TickNets project, found: {candidates}')
    return candidates[0]

def matching_checkpoints(search_root):
    matches = []
    for path in search_root.rglob('last.pt'):
        try:
            checkpoint = torch.load(path, map_location='cpu', weights_only=True)
        except Exception:
            continue
        config = checkpoint.get('config', {})
        if (checkpoint.get('epoch') == 100 and config.get('model') == 'c'
                and config.get('variant') == 'Mid224' and config.get('epochs') == 200
                and 'rng_state' in checkpoint and (path.parent / 'epochs.csv').is_file()):
            matches.append(path)
    return matches

PROJECT_ROOT = locate_project()
CHECKPOINTS = matching_checkpoints(Path('/kaggle/input'))
if not CHECKPOINTS:
    archives = list(Path('/kaggle/input').rglob('c_mid224_seed42_epoch100.zip'))
    for index, archive in enumerate(archives):
        shutil.unpack_archive(archive, Path(f'/kaggle/working/c_epoch100_input_{index}'))
    CHECKPOINTS = matching_checkpoints(Path('/kaggle/working'))
if len(CHECKPOINTS) != 1:
    raise RuntimeError(f'Expected exactly one C/Mid224 epoch-100 checkpoint, found: {CHECKPOINTS}')
RESUME_CHECKPOINT = CHECKPOINTS[0]
DATA_ROOT = PROJECT_ROOT / 'data'
OUTPUT_DIR = Path('/kaggle/working/runs/c_mid224_seed42')
print('Project:', PROJECT_ROOT)
print('Resume:', RESUME_CHECKPOINT)
print('Output:', OUTPUT_DIR)

Project: /kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets
Resume: /kaggle/input/datasets/lastapple/c-mid224-seed42-epoch100/c_mid224_seed42/last.pt
Output: /kaggle/working/runs/c_mid224_seed42


In [4]:
if OUTPUT_DIR.exists():
    raise FileExistsError(f'Output already exists: {OUTPUT_DIR}')
command = [
    sys.executable, str(PROJECT_ROOT / 'train_mid.py'),
    '--data-root', str(DATA_ROOT), '--variant', 'Mid224', '--model', 'c',
    '--seed', '42', '--epochs', '200', '--batch-size', '64', '--num-workers', '0',
    '--device', 'auto', '--resume', str(RESUME_CHECKPOINT),
    '--output-dir', str(OUTPUT_DIR),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

Running: /usr/bin/python3 /kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets/train_mid.py --data-root /kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets/data --variant Mid224 --model c --seed 42 --epochs 200 --batch-size 64 --num-workers 0 --device auto --resume /kaggle/input/datasets/lastapple/c-mid224-seed42-epoch100/c_mid224_seed42/last.pt --output-dir /kaggle/working/runs/c_mid224_seed42
{"epoch": 101, "learning_rate": 0.050000000000000024, "train_loss": 0.083706320002079, "train_top1": 96.784, "train_samples": 25000}
{"epoch": 102, "learning_rate": 0.049214634134409, "train_loss": 0.07023664527654648, "train_top1": 97.428, "train_samples": 25000}
{"epoch": 103, "learning_rate": 0.04842946204609361, "train_loss": 0.06931697179913521, "train_top1": 97.428, "train_samples": 25000}
{"epoch": 104, "learning_rate": 0.04764467746451789, "train_loss": 0.07193953026771545, "train_top1": 97.348, "train_samples": 25000}
{"epoch": 105, "lear

CompletedProcess(args=['/usr/bin/python3', '/kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets/train_mid.py', '--data-root', '/kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets/data', '--variant', 'Mid224', '--model', 'c', '--seed', '42', '--epochs', '200', '--batch-size', '64', '--num-workers', '0', '--device', 'auto', '--resume', '/kaggle/input/datasets/lastapple/c-mid224-seed42-epoch100/c_mid224_seed42/last.pt', '--output-dir', '/kaggle/working/runs/c_mid224_seed42'], returncode=0)

In [5]:
checkpoint = torch.load(OUTPUT_DIR / 'last.pt', map_location='cpu', weights_only=True)
assert checkpoint['epoch'] == 200
assert checkpoint['config']['model'] == 'c'
assert checkpoint['config']['variant'] == 'Mid224'
metrics = json.loads((OUTPUT_DIR / 'test_metrics.json').read_text(encoding='utf-8'))
with (OUTPUT_DIR / 'epochs.csv').open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 200 and [int(row['epoch']) for row in rows] == list(range(1, 201))
archive = shutil.make_archive('/kaggle/working/c_mid224_seed42_final', 'zip',
                               root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print(json.dumps({'status': 'complete', 'epoch': 200, 'test_metrics': metrics,
                  'run': str(OUTPUT_DIR), 'download_archive': archive}, indent=2))

{
  "status": "complete",
  "epoch": 200,
  "test_metrics": {
    "loss": 0.16260036659240723,
    "top1": 94.4,
    "samples": 250
  },
  "run": "/kaggle/working/runs/c_mid224_seed42",
  "download_archive": "/kaggle/working/c_mid224_seed42_final.zip"
}
